# Report: Movie Recommendation System

## Introduction / Problem Statement

The overall goal of the project was to develop a movie recommendation system that returns five relevant recommendations given one input movie. The task was treated as a ranking problem rather than a classification or regression problem, since the objective was to order candidate movies by relevance. The dataset used was MovieLens `ml-latest`, which contains movies, user ratings, tags, and links to external movie databases.

Two recommendation strategies were considered relevant: content-based filtering and collaborative filtering. Content-based filtering relies on movie metadata, while collaborative filtering relies on similarities in user behavior. A hybrid model was selected as the final solution because the two approaches complement each other: content-based filtering provides broader coverage, while collaborative filtering captures behavioral similarity that is not directly visible in the metadata. In the present model, the content-based component was built with TF-IDF and cosine similarity over genres and tags. TF-IDF represents each movie as a weighted text vector, and cosine similarity measures how similar two such vectors are. The collaborative component was built with item-based k-nearest neighbors on a movie-user rating matrix, where movies are treated as similar if they show similar rating patterns across users.

The final recommendation score was computed as a weighted combination of normalized signals from the content-based model, the collaborative model, and additional movie-level signals. Because the problem is fundamentally a ranking task, model performance was evaluated with NDCG@5 rather than standard accuracy. A ranking-based metric was required because the quality of the highest-ranked recommendations was more important than predicting a single correct label.

## Data Analysis (EDA)

The exploratory analysis showed that the rating data is both large and highly sparse. Most movies have relatively few ratings, while a small number of popular movies account for a large share of the interactions. User activity is also unevenly distributed, with many users contributing only a limited number of ratings and a small group of highly active users contributing a substantial amount of data.

These patterns affected the modeling choices directly. Since collaborative filtering depends on stable interaction data, ratings were filtered more aggressively for the collaborative component than for the content-based component. This reduced noise in the item-user matrix while still allowing less frequently rated movies to remain available in the metadata-based part of the system. The dataset also did not contain useful demographic variables such as age or occupation in the chosen setup, which made it more natural to rely on behavioral signals such as rating activity, rating count, and expert users.

The analysis also showed that genres alone were too limited to capture meaningful similarity between movies. Tags provided a richer and more descriptive signal, and were therefore combined with genres into a shared text representation. In addition, user activity patterns made it possible to define highly active users as expert users, which later became an additional signal in the hybrid ranking model. The strong skew in rating counts also motivated the use of quantile-based popularity groups instead of fixed thresholds, since fixed cutoffs produced less informative groupings.

## Model

Several recommendation approaches were considered and tested, including simpler content-based and collaborative baselines (found in the folder "baseline_models"), but the final system was implemented as a hybrid model. A shared feature table was first constructed from movies, tags, links, and filtered ratings. This table contained searchable titles, release year, textual movie features, rating statistics, expert-based statistics, and external ids. Genres and tags were combined into a single text field, `text_features`, which was used by the content-based component.

The content-based model represented each movie with TF-IDF and measured similarity with cosine similarity. The collaborative model represented each movie in an item-user rating matrix and used item-based k-nearest neighbors with cosine distance. For each query movie, candidate sets were generated from both components and merged into a common candidate table. Missing scores were filled with zero, after which movie-level features such as average rating and expert mean rating were added. Expert users were defined from the highest activity group, and their movie ratings were summarized into an expert mean rating feature. A new-movie expert signal was also included by combining expert support with release year information.

The final ranking was based on normalized candidate scores. In the selected configuration, the model used the weights  
- $w_c = 0.4$ for collaborative score,  
- $w_t = 0.35$ for content score,  
- $w_r = 0.1$ for average rating,  
- $w_e = 0.1$ for expert mean rating,  
- $w_n = 0.05$ for a new-movie expert signal,  
- $w_p = 0.1$ for popularity group.    

This gave the final score

$\text{final score} = w_c s_c + w_t s_t + w_r s_r + w_e s_e + w_n s_n + w_p s_p$.

A popularity-based signal was also included in the final model. The signal was derived from quantile-based popularity groups built from movie rating counts, where the groups low, medium, high, and very high were mapped to an ordinal popularity score. During development, the model was also optimized by reducing candidate set sizes and by introducing a direct movie-to-row index mapping for the TF-IDF matrix.

Model performance was evaluated with NDCG@5 (Normalized Discounted Cumulative Gain) by comparing the recommended movies for a query film with sets of relevant movies derived from user preference patterns. Several weight configurations were tested in order to compare more content-oriented, more balanced, and more collaborative-oriented versions of the hybrid model. NDCG measures how well relevant movies are placed near the top of the recommendation list, and also gives less credit to relevant items that appear lower down. Thus it makde it suitable for this system, where the top few suggestions are more important than the full ranking.

## Results

The results showed that collaborative-heavy configurations performed best among the tested alternatives, which indicates that behavioral similarity carried a strong recommendation signal in the filtered MovieLens data. At the same time, the content-based component still contributed useful support and broader coverage. The selected final model therefore retained both components, but with slightly higher weight on the collaborative part. Diagnostic outputs were also used during development to inspect unusual recommendation lists and understand how the different score components affected the final ranking.

A popularity-based signal was also tested as an additional ranking feature. This signal was based on the number of ratings per movie, which was used as a proxy for movie popularity. Instead of using fixed thresholds, movies were divided into quantile-based popularity groups using the 50th, 90th, and 99th percentiles of the rating count distribution. This produced four groups: low, medium, high, and very high popularity.

A quantile-based grouping was preferred because the distribution of rating counts was highly skewed. Most movies had relatively few ratings, while a small number of movies had very many ratings. Fixed cutoffs therefore gave unbalanced groups, whereas quantile-based thresholds made it easier to define groups that better reflected the structure of the data.

The purpose of this popularity-based signal was to test whether ranking could be improved by including information about how widely a movie had been rated. In the final comparison, the popularity-based variants improved NDCG@5 compared with the plain more-collaborative baseline, and the best result was obtained with a popularity weight of 0.10. For that reason, popularity was retained in the selected final model.

## Discussion

The results suggest that a hybrid recommendation model is a suitable solution for this task. The collaborative component contributed the strongest signal in the final evaluation, but the content-based component remained important for coverage and for movies with weaker collaborative support. The final model therefore benefited from combining both approaches rather than relying on only one of them.

The project also showed the value of evaluating design decisions quantitatively instead of relying only on intuition. In the final evaluation, the popularity-based signal improved NDCG when it was added in a controlled way on top of the hybrid model, and it was therefore retained in the selected configuration.

The model was also relatively expensive to run compared with a simple baseline, especially during the first run when preprocessing artifacts had to be built. For that reason, caching was introduced for the shared feature table, TF-IDF artifacts, and collaborative artifacts. This improved runtime substantially and made the application more practical, although it also introduced a dependency on rebuilding cached artifacts when the environment or model structure changes.

Some limitations remain. The rating matrix is sparse, which makes collaborative filtering sensitive to filtering choices and cold-start cases. Movies with very limited metadata or rating history may still produce weak recommendations or none at all. In addition, the application layer depends on external poster data if images are enabled. Further work could therefore focus on stronger handling of sparse movies, more efficient precomputation, and additional methods for increasing recommendation diversity.

Another possible extension would have been to investigate whether clustering could improve either the efficiency or the diversity of the recommendation process. For example, clustering movies in the content-based or hybrid feature space could make it possible to restrict candidate generation to smaller and more coherent subsets of movies, which may reduce runtime. Clustering could also be used to increase diversity in the final recommendation list by selecting candidates from different regions of the feature space. This approach was not implemented in the final system, but it would be a relevant direction for further work.  
  



---

External sources: https://www.evidentlyai.com/ranking-metrics/ndcg-metric